# First Implementation of the RSA model 

In [ ]:
import numpy as np 
import random as rd
import spacy
import random as rd
import json 
import sys 
import logging
sys.path.insert(1, '/home/user/Documents/Sprachgebrauchsmodellierung_RSA/Hausarbeit_RSA/repo/scripts/')
from gamestate import gs
nlp = spacy.load('de_core_news_sm')


In [2]:
print(gs.V)

['Notwendigkeit', 'Zugang', 'Glanz', 'Band', 'Online', 'Fussballer', 'Filter', 'Erholung', 'U00', 'Produkt']


In [48]:
class Agent:
    def __init__(self, player, knows_imp=False): 
         self.player = player
         self.role = gs.ROLES[player]
         self.target_imp = self.get_target()
         self.alpha = gs.RATIONALITY[player]
         if knows_imp: 
            self.priors = {}
            for p in gs.PLAYERS:
                if p==self.player:
                    self.priors[p]= self.get_priors()
                elif gs.ROLES[p]=="agent":
                    self.priors[p]= self.get_priors(knows_imp=False)
                else:
                    self.priors[p]= self.get_priors(knows_t=False)


    def get_target(self):
        max_freq = max(gs.FREQ.values())
        res = [k for k in gs.FREQ if gs.FREQ[k] == max_freq]
        return(rd.choice(res))
    
    def get_priors(self, knows_t = True, knows_imp = True):
        priors = []
        # P1 weiß wer der Imposter ist 
        for p, w in gs.W: 
            if knows_imp: 
                if gs.ROLES[p]=="agent":
                    priors.append(0)
                else:
                    if knows_t: 
                        if w==gs.TARGET:
                            priors.append(1)
                        else:
                            priors.append(0)
                    else:
                        priors.append(gs.FREQ[w])
            else:
                if p==self.player:
                    priors.append(0)
                else:
                    if w==gs.TARGET:
                        priors.append(1/(len(gs.PLAYERS)-1))
                    else:
                        priors.append(0)
                    #priors.append(1/(len(gs.PLAYERS)-1))
                    
        priors = [w/sum(priors) for w in priors]
        priors = np.array(priors)
        #print(f"priors shape: {priors.shape}, players: {gs.PLAYERS}, V: {gs.V}")
        priors.shape = (len(gs.PLAYERS),len(gs.V))
        return(priors)
    
    def get_P_u_w(self,u,w,t_imp,speaker="P1"):
        imp_ind, t_ind = w
        if gs.PLAYERS[imp_ind]==speaker: 
            P_u_w = gs.SIMS[t_ind, u]
        else:
            P_u_w = gs.SIMS[t_imp, u]
        return(P_u_w)

    def L0(self, u, w, listener="P2", speaker="P1"):
        imp, t = w
        imp = gs.PLAYERS.index(imp)
        t = gs.V.index(t)
        u = gs.V.index(u)
        t_imp = gs.V.index(self.target_imp)
        priors_l = self.priors[listener]

        print(f"P(u|w) in L(0): {self.get_P_u_w(u,(imp,t),t_imp) }, priors: {priors_l[imp][t]}")
        P_w_u = self.get_P_u_w(u,(imp,t),t_imp, speaker=speaker) * priors_l[imp][t]
        print(f"P(w|u) in L(0): {P_w_u}")
        SUM = np.nansum([self.get_P_u_w(u,(gs.PLAYERS.index(w_n[0]), gs.V.index(w_n[1])),t_imp, speaker=speaker) * priors_l[gs.PLAYERS.index(w_n[0]),gs.V.index(w_n[1])] for w_n in gs.W])
        print("SUM in L0:", SUM)
        P_w_u = P_w_u/SUM
        print(f"normalized P(w|u) in L(0): {P_w_u}")

        return(P_w_u)
    
    def utility(self, u, w, partner="P2", speaker="P1"):
        #print(f"listen(u,w,listener=partner): {listen(u,w,listener=partner)}, listen(u,(P3,t),listener=imp): {listen(u,(imp,w[1]),listener=imp)}")
        # Anwendung von log-Funktion auf Differenz beruht ausschließlich auf ausprobieren
        #print(f"partner part of utility:\n with logs: {np.log(self.L0(u,w,listener=partner))}, without logs:{self.L0(u,w,listener=partner)}\n imposter part of utility:\n with logs: {np.log(self.L0(u,(imp,w[1]),listener=imp))}, without logs:{self.L0(u,(imp,w[1]),listener=imp)}\n")
        U = np.log(self.L0(u,w,listener=partner, speaker=speaker)) #+ np.log(self.L0(u,(imp,w[1]),listener=imp)) # weil auf logarithmischer Skala negativ 
        #SUM = np.nansum([np.log(listen(u_n,w,listener=partner)) - np.log(listen(u_n,w,listener=imp)) for u_n in range(len(V))])
        print(f"utility of word {u}: {U}")
        return(U)
    
    def S1(self, u,w, alpha=1, speaker="P1"): 
        #print(f"P(w|u) in S1: {np.exp(alpha * self.utility(u,w))}, U in S1: {self.utility(u,w)}")
        P_u_w = np.exp(alpha * self.utility(u,w, speaker=speaker))
        SUM = np.nansum([np.exp(alpha * self.utility(u_n, w, speaker=speaker)) for u_n in gs.V])
        P_u_w = P_u_w/SUM
        return(P_u_w)
    
    def L1(self, u,w,alpha=1,speaker="P2", listener="P1"):
        imp, t = w
        imp_ind = gs.PLAYERS.index(imp)
        t_ind = gs.V.index(t)
        # WHAT IS THE DIFFERENCE
        if imp==speaker:
            P_w_u = self.S1(u,w,alpha=alpha) * self.priors[listener][imp_ind][t_ind]
        else:
            print(f"P(u|w): {self.S1(u,w,alpha=alpha)}, prior: {self.priors[listener][imp_ind][t_ind]}")
            P_w_u = self.S1(u,w,alpha=alpha) * self.priors[listener][imp_ind][t_ind]
        print("P(w|u) before normalization: ", P_w_u)
        SUM = np.nansum([self.S1(u,w_n,alpha=alpha) * self.priors[listener][gs.PLAYERS.index(w_n[0])][gs.V.index(w_n[1])] for w_n in gs.W])
        print("probability sum: ", SUM)

        P_w_u = P_w_u/SUM
        return(P_w_u)



In [44]:
print(gs.TARGET)
print(gs.V.index(gs.TARGET))
print(P1.target_imp)
print(gs.PLAYERS)

Band
3
Online
['P1', 'P2', 'P3']


In [49]:
P1 = Agent("P1", knows_imp=True)


In [47]:
print(gs.V)

['Notwendigkeit', 'Zugang', 'Glanz', 'Band', 'Online', 'Fussballer', 'Filter', 'Erholung', 'U00', 'Produkt']


In [50]:
P1.S1(gs.V[8], ("P3", gs.TARGET))

P(u|w) in L(0): 0.21946340799331665, priors: 0.5
P(w|u) in L(0): 0.10973170399665833
SUM in L0: 0.21946340799331665
normalized P(w|u) in L(0): 0.5
utility of word U00: -0.6931471805599453
P(u|w) in L(0): 0.3650882840156555, priors: 0.5
P(w|u) in L(0): 0.18254414200782776
SUM in L0: 0.3650882840156555
normalized P(w|u) in L(0): 0.5
utility of word Notwendigkeit: -0.6931471805599453
P(u|w) in L(0): 0.2818779945373535, priors: 0.5
P(w|u) in L(0): 0.14093899726867676
SUM in L0: 0.2818779945373535
normalized P(w|u) in L(0): 0.5
utility of word Zugang: -0.6931471805599453
P(u|w) in L(0): 0.37178704142570496, priors: 0.5
P(w|u) in L(0): 0.18589352071285248
SUM in L0: 0.37178704142570496
normalized P(w|u) in L(0): 0.5
utility of word Glanz: -0.6931471805599453
P(u|w) in L(0): 0.4384180009365082, priors: 0.5
P(w|u) in L(0): 0.2192090004682541
SUM in L0: 0.4384180009365082
normalized P(w|u) in L(0): 0.5
utility of word Band: -0.6931471805599453
P(u|w) in L(0): 0.0, priors: 0.5
P(w|u) in L(0): 0.

/tmp/ipykernel_7985/2277381829.py:75: RuntimeWarning: invalid value encountered in scalar divide
  P_w_u = P_w_u/SUM


np.float64(0.1111111111111111)

In [24]:
P1.L1(gs.V[2], ("P3",gs.TARGET), listener="P2")

P(u|w) in L(0): 0.37178704142570496, priors: 0.5
SUM in L0: 0.37178704142570496
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.3650882840156555, priors: 0.5
SUM in L0: 0.3650882840156555
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.2818779945373535, priors: 0.5
SUM in L0: 0.2818779945373535
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.37178704142570496, priors: 0.5
SUM in L0: 0.37178704142570496
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.4384180009365082, priors: 0.5
SUM in L0: 0.4384180009365082
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.0, priors: 0.5
SUM in L0: 0.0
P(w|u) from L0: nan
P(u|w) in L(0): 0.9999998807907104, priors: 0.5
SUM in L0: 0.9999998807907104
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.21370074152946472, priors: 0.5
SUM in L0: 0.21370074152946472
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.2617661952972412, priors: 0.5
SUM in L0: 0.2617661952972412
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.21946340799331665, priors: 0.5
SUM in L0: 0.21946340799331665
P(w|u) from L0: 0.5
P(u|w) in L(0): 0.236378014087677, priors:

/tmp/ipykernel_7985/849310866.py:73: RuntimeWarning: invalid value encountered in scalar divide
  P_w_u = P_w_u/SUM
/tmp/ipykernel_7985/849310866.py:81: RuntimeWarning: divide by zero encountered in log
  U = np.log(self.L0(u,w,listener=partner)) #+ np.log(self.L0(u,(imp,w[1]),listener=imp)) # weil auf logarithmischer Skala negativ
/tmp/ipykernel_7985/849310866.py:89: RuntimeWarning: invalid value encountered in scalar divide
  P_u_w = P_u_w/SUM


np.float64(0.5)

In [39]:
print(gs.TARGET)
print(P1.target_imp)

Band
Online


In [43]:
for u in gs.V:
    p_w_u = P1.S1(u,("P3",gs.TARGET))
    print(f"u: {u}, P(w|u): {p_w_u}")

utility of word Notwendigkeit: -0.6931471805599453
utility of word Notwendigkeit: -0.6931471805599453
utility of word Zugang: -0.6931471805599453
utility of word Glanz: -0.6931471805599453
utility of word Band: -0.6931471805599453
utility of word Online: nan
utility of word Fussballer: -0.6931471805599453
utility of word Filter: -0.6931471805599453
utility of word Erholung: -0.6931471805599453
utility of word U00: -0.6931471805599453
utility of word Produkt: -0.6931471805599453
u: Notwendigkeit, P(w|u): 0.1111111111111111
utility of word Zugang: -0.6931471805599453
utility of word Notwendigkeit: -0.6931471805599453
utility of word Zugang: -0.6931471805599453
utility of word Glanz: -0.6931471805599453
utility of word Band: -0.6931471805599453
utility of word Online: nan
utility of word Fussballer: -0.6931471805599453
utility of word Filter: -0.6931471805599453
utility of word Erholung: -0.6931471805599453
utility of word U00: -0.6931471805599453
utility of word Produkt: -0.6931471805599

/tmp/ipykernel_7985/797096227.py:73: RuntimeWarning: invalid value encountered in scalar divide
  P_w_u = P_w_u/SUM


In [14]:
print(gs.ROLES)

{'P1': 'agent', 'P2': 'agent', 'P3': 'imposter'}


In [18]:
for k,v in P1.priors.items():
    print(f"{k}: {v}")

P1: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]]
P2: [[0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.5 0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.5 0.  0.  0.  0.  0.  0. ]]
P3: [[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.02902473 0.07805299 0.0196113  0.22553    0.47067129 0.0138848
  0.0241219  0.01829735 0.05275441 0.06805122]]


## next steps 

- implement cosine similarity 
- think about how to adjust probabilities with L1 functions 


In [33]:
with open('../output/random_nouns.json', 'r') as file:
    rd_100 = json.load(file)

print(rd_100)


['Ball', 'Künstler', 'Spaziergang', 'Partnerin', 'Bayer', 'Ereignis', 'Feuerwehr', 'Petition', 'Bundestag', 'Erfüllung', 'Ausweis', 'Rückkehr', 'Zitat', 'Verfasst', 'Schülerin', 'Wette', 'Aluminium', 'Saal', 'Einbezog', 'Eign', 'Klinik', 'Playstation', 'Nutze', 'Zufriedenheit', 'Opposition', 'Schweine', 'Messung', 'Steigerung', 'Hafen', 'Service', 'Zwischenzeit', 'Kenntnis', 'Rest', 'Terrorismus', 'Ulm', 'Wettbewerb', 'Freund', 'Element', 'Anwohner', 'Bgb', 'Empfänger', 'Rufe', 'Station', 'Milde', 'Gefasst', 'Invasion', 'Herauszufinde', 'Gewalt', 'Realschule', 'Solidarität', 'Produkt', 'Koffer', 'Mann', 'Verbrechen', 'Weisheit', 'Kontinent', 'Komplex', 'Pressemitteilung', 'Vorteil', 'Doktor', 'Symbol', 'Tasse', 'Scheidung', 'Stürmer', 'Statistik', 'Christ', 'Rundschau', 'Furcht', 'Dialog', 'Partie', 'Nation', 'Haus', 'Jungfrau', 'Tiger', 'Zieh', 'Rufe', 'Hochschule', 'Vorwurf', 'Rechtsprechung', 'Hotel', 'Fahrt', 'Wahrheit', 'Meinung', 'Verkehrt', 'Zwischenzeitlich', 'Einleitung', 'Nac

In [ ]:
class Imposter:
    def __init__(self, player): 
         self.player = player
         self.role = roles[player]
         self.target = self.get_target()
         self.P_t = freq.copy()
         self.alpha = rationality[player]
         self.target_imp = self.get_target()

    def get_target(self):
        mv = max(freq.values())
        res = [k for k in freq if freq[k] == mv]
        return(rd.choice(res))
    
    
    def L0(self, u, t, p_player = [2/3, 1/3]):
        p_agent = get_similarity(u,t) * p_player[0]
        p_imposter = get_similarity(u,self.target_imp) * p_player[1]
        p_agent_norm = p_agent/(p_agent + p_imposter)
        p_imposter_norm = p_imposter/(p_agent + p_imposter)
        return((p_agent_norm, p_imposter_norm))

    def S1(self, t):
        p_distr = []
        for u in V: 
            # what exactly is the world? The persona the player wants to communicate? That would always be agent.
            p_u = np.exp(self.alpha * (np.log(self.L0(u, t)[0])- C[u]))
            #print("p_u:", p_u)
            p_distr.append(p_u)

        p_sum = sum(p_distr)
        u_dic = {u: p/p_sum for u,p in zip(V,p_distr)}
        #print(p_distr)
        #print(u_dic)
        return(u_dic)

    def choose_u(self): 
        P_u = self.S1(self.target)
        mv = max(P_u.values())
        res = [k for k in P_u if P_u[k] == mv]
        return(rd.choice(res))
    

    def L1(self, u):
        t_distr = []
        for t in V: 
            p = self.S1(t)[u] * self.P_t[t]
            t_distr.append(p)

        p_sum = sum(t_distr)
        t_dic = {t: p/p_sum for t,p in zip(V,t_distr)}

        # is the new probability distribution a mix of the old distribution and the newly calculated one or is it just the new one? 
        self.P_t = t_dic

        return(t_dic)
    



class GameState:
    def __init__(self): 
        self.previous_words = []

    def say_word(self, word):
        self.previous_words += word